In [46]:
import os
import numpy as np
import pandas as pd
import yfinance as yf

# Path dataset
DATA_RAW_DIR = "../data/raw"
DATA_PROCESSED_DIR = "../data/processed"
os.makedirs(DATA_RAW_DIR, exist_ok=True)
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

tickers = ["BTC-USD", "ETH-USD", "BNB-USD"]
start = "2015-01-01"
end = None # None = sampai data terbaru
interval = "1d"

# untuk penamaan file
run_end_date = pd.Timestamp.today().date().isoformat()

In [47]:
# Download data raw
df = yf.download(
    tickers=tickers,
    start=start,
    end=end,
    interval=interval,
    group_by="ticker",
    auto_adjust=False,
    threads=True
)

df.tail()

[*********************100%***********************]  3 of 3 completed


Ticker           BTC-USD                                            \
Price               Open          High           Low         Close   
Date                                                                 
2025-12-15  88171.078125  89983.921875  85304.078125  86419.781250   
2025-12-16  86424.406250  88170.093750  85381.687500  87843.984375   
2025-12-17  87847.617188  90264.570312  85316.265625  86143.757812   
2025-12-18  86144.367188  89412.664062  84436.312500  85462.507812   
2025-12-19  85463.296875  88711.984375  85134.062500  88305.687500   

Ticker                                     ETH-USD                            \
Price          Adj Close       Volume         Open         High          Low   
Date                                                                           
2025-12-15  86419.781250  45559514323  3060.481445  3175.118164  2899.685791   
2025-12-16  87843.984375  41262178223  2964.379639  2978.921387  2890.011719   
2025-12-17  86143.757812  44243392914  2964.019531  3025.815430  2791.333984   
2025-12-18  85462.507812  52667115348  2831.488281  2994.552734  2777.119629   
2025-12-19  88305.687500  56295452672  2826.969727  2986.404785  2811.689209   

Ticker                                                 BNB-USD              \
Price             Close    Adj Close        Volume        Open        High   
Date                                                                         
2025-12-15  2964.183105  2964.183105  2.876598e+10  877.789246  894.205261   
2025-12-16  2964.180908  2964.180908  2.270919e+10  858.545898  877.613098   
2025-12-17  2831.403564  2831.403564  2.615983e+10  874.905518  875.110840   
2025-12-18  2827.069092  2827.069092  2.919418e+10  842.931274  850.898376   
2025-12-19  2982.984375  2982.984375  3.420553e+10  829.286743  851.640503   

Ticker                                                        
Price              Low       Close   Adj Close        Volume  
Date                                                          
2025-12-15  843.767883  858.545105  858.545105  2.222158e+09  
2025-12-16  851.367615  874.899048  874.899048  2.379705e+09  
2025-12-17  834.851685  842.930176  842.930176  2.968394e+09  
2025-12-18  819.983826  829.301147  829.301147  3.195634e+09  
2025-12-19  823.339172  850.505981  850.505981  5.956272e+09

In [ ]:

# convert ke long format (RAW long)

if isinstance(df.columns, pd.MultiIndex):
    # Hasil multi-ticker: jadikan long => Date, Ticker, Open, High, Low, Close, Adj Close, Volume
    df_long = (
        df.stack(level=0)
        .rename_axis(["Date", "Ticker"])
        .reset_index()
        .sort_values(["Ticker", "Date"])
        .reset_index(drop=True)
    )
else:
    # fallback kalau cuma 1 ticker
    df_long = df.reset_index()
    df_long.insert(1, "Ticker", tickers[0])

df_long["Date"] = pd.to_datetime(df_long["Date"], errors="coerce")
df_long = df_long.dropna(subset=["Date"]).copy()
df_long = df_long.sort_values(["Ticker", "Date"]).reset_index(drop=True)

df_long.head()

C:\Users\asusg\AppData\Local\Temp\ipykernel_23912\3138281168.py:6: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack(level=0)


Price,Date,Ticker,Open,High,Low,Close,Adj Close,Volume
0,2017-11-09,BNB-USD,2.05314,2.17423,1.89394,1.99077,1.99077,19192200.0
1,2017-11-10,BNB-USD,2.00773,2.06947,1.64478,1.79684,1.79684,11155000.0
2,2017-11-11,BNB-USD,1.78628,1.91775,1.61429,1.67047,1.67047,8178150.0
3,2017-11-12,BNB-USD,1.66889,1.67280,1.46256,1.51969,1.51969,15298700.0
4,2017-11-13,BNB-USD,1.52601,1.73502,1.51760,1.68662,1.68662,12238800.0


In [ ]:
# cek adjclose dan close 
if "Adj Close" in df_long.columns:
    diff_abs = (df_long["Adj Close"] - df_long["Close"]).abs()

    print("Adj Close vs Close:")
    print("  max abs diff :", float(diff_abs.max()))
    print("  mean abs diff:", float(diff_abs.mean()))
    print("  count diff>0 :", int((diff_abs > 0).sum()))

    # tampilkan contoh baris yang beda (kalau ada)
    contoh_beda = df_long.loc[diff_abs > 0, ["Date", "Ticker", "Close", "Adj Close"]].head(10)
    if len(contoh_beda) == 0:
        print("Tidak ada perbedaan (semua sama).")
    else:
        print("Ada perbedaan, contoh:")
        display(contoh_beda)
else:
    print("Kolom 'Adj Close' tidak ditemukan.")

Adj Close vs Close:
  max abs diff : 0.0
  mean abs diff: 0.0
  count diff>0 : 0
Tidak ada perbedaan (semua sama).


In [50]:
# drop_adjclose_and_tidy_columns (RAW)
# Drop Adj Close (karena sudah terbukti sama dengan Close)
if "Adj Close" in df_long.columns:
    df_long = df_long.drop(columns=["Adj Close"])

# Pastikan numerik (aman kalau ada yang kebaca string)
for c in ["Open", "High", "Low", "Close", "Volume"]:
    if c in df_long.columns:
        df_long[c] = pd.to_numeric(df_long[c], errors="coerce")

# Tidy kolom (RAW cukup OHLCV)
raw_cols = ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]
df_long = df_long[[c for c in raw_cols if c in df_long.columns]].copy()

df_long.head()

Price,Date,Ticker,Open,High,Low,Close,Volume
0,2017-11-09,BNB-USD,2.05314,2.17423,1.89394,1.99077,19192200.0
1,2017-11-10,BNB-USD,2.00773,2.06947,1.64478,1.79684,11155000.0
2,2017-11-11,BNB-USD,1.78628,1.91775,1.61429,1.67047,8178150.0
3,2017-11-12,BNB-USD,1.66889,1.67280,1.46256,1.51969,15298700.0
4,2017-11-13,BNB-USD,1.52601,1.73502,1.51760,1.68662,12238800.0


In [51]:
print("Tickers in df_long:", df_long["Ticker"].unique())

summary = df_long.groupby("Ticker")["Date"].agg(["min", "max", "count"])
print(summary)

print("\nMissing Date:", df_long["Date"].isna().sum())
print("Missing Close:", df_long["Close"].isna().sum())
print("Duplicate (Ticker, Date):", df_long.duplicated(["Ticker", "Date"]).sum())

# cek missing days per ticker (idealnya 0 untuk crypto 1d)
for t in df_long["Ticker"].unique():
    dft = df_long[df_long["Ticker"] == t].sort_values("Date")
    expected = pd.date_range(dft["Date"].min(), dft["Date"].max(), freq="D")
    missing = expected.difference(dft["Date"])
    print(f"{t} missing days: {len(missing)}")
    if len(missing) > 0:
        print("  contoh missing:", list(missing[:10]))

Tickers in df_long: ['BNB-USD' 'BTC-USD' 'ETH-USD']
               min        max  count
Ticker                              
BNB-USD 2017-11-09 2025-12-19   2963
BTC-USD 2015-01-01 2025-12-19   4006
ETH-USD 2017-11-09 2025-12-19   2963

Missing Date: 0
Missing Close: 0
Duplicate (Ticker, Date): 0
BNB-USD missing days: 0
BTC-USD missing days: 0
ETH-USD missing days: 0


In [52]:
# save dataset RAW ke data/raw
raw_path = os.path.join(
    DATA_RAW_DIR,
    f"crypto_yahoo_{interval}_{start}_to_{run_end_date}_LONG.csv"
)

df_long.to_csv(raw_path, index=False, date_format="%Y-%m-%d")
raw_path

'../data/raw\\crypto_yahoo_1d_2015-01-01_to_2025-12-19_LONG.csv'

In [ ]:

# build PROCESSED common range
# intersection semua aset karena aset ETH dan BNB hanya tersedia dari 2017-11-09
common_start = df_long.groupby("Ticker")["Date"].min().max()
common_end   = df_long.groupby("Ticker")["Date"].max().min()

print("Common start:", common_start.date())
print("Common end  :", common_end.date())

df_common = df_long[(df_long["Date"] >= common_start) & (df_long["Date"] <= common_end)].copy()
df_common = df_common.sort_values(["Ticker", "Date"]).reset_index(drop=True)

df_common.groupby("Ticker")["Date"].agg(["min", "max", "count"])

Common start: 2017-11-09
Common end  : 2025-12-19


,min,max,count
Ticker,,,
BNB-USD,2017-11-09,2025-12-19,2963
BTC-USD,2017-11-09,2025-12-19,2963
ETH-USD,2017-11-09,2025-12-19,2963


In [54]:
# align to full daily calendar (PROCESSED aligned)
# kalau tidak ada missing day, hasilnya sama
def align_daily(df_in: pd.DataFrame) -> pd.DataFrame:
    out = []
    for t, g in df_in.groupby("Ticker"):
        g = g.sort_values("Date").set_index("Date")
        full_idx = pd.date_range(g.index.min(), g.index.max(), freq="D")
        g = g.reindex(full_idx)
        g["Ticker"] = t
        g = g.reset_index().rename(columns={"index": "Date"})
        out.append(g)
    return pd.concat(out, ignore_index=True)

df_aligned = align_daily(df_common)

print("Missing Close after align:", df_aligned["Close"].isna().sum())
print("Duplicate (Ticker, Date) after align:", df_aligned.duplicated(["Ticker", "Date"]).sum())

df_aligned.head()

Missing Close after align: 0
Duplicate (Ticker, Date) after align: 0


Price,Date,Ticker,Open,High,Low,Close,Volume
0,2017-11-09,BNB-USD,2.05314,2.17423,1.89394,1.99077,19192200.0
1,2017-11-10,BNB-USD,2.00773,2.06947,1.64478,1.79684,11155000.0
2,2017-11-11,BNB-USD,1.78628,1.91775,1.61429,1.67047,8178150.0
3,2017-11-12,BNB-USD,1.66889,1.67280,1.46256,1.51969,15298700.0
4,2017-11-13,BNB-USD,1.52601,1.73502,1.51760,1.68662,12238800.0


In [55]:
# feature engineering (PROCESSED features)
# LogClose, LogReturn (per ticker)
df_feat = df_aligned.copy()

# pastikan numerik
for c in ["Open", "High", "Low", "Close", "Volume"]:
    if c in df_feat.columns:
        df_feat[c] = pd.to_numeric(df_feat[c], errors="coerce")

# fitur
df_feat["LogClose"] = np.where(df_feat["Close"] > 0, np.log(df_feat["Close"]), np.nan)
df_feat["LogReturn"] = df_feat.groupby("Ticker")["LogClose"].diff()
df_feat["Return"] = df_feat.groupby("Ticker")["Close"].pct_change()

# rapikan kolom final processed
processed_cols = ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume", "LogClose", "LogReturn", "Return"]
df_common = df_common[[c for c in ["Date","Ticker","Open","High","Low","Close","Volume"] if c in df_common.columns]].copy()
df_aligned = df_aligned[[c for c in ["Date","Ticker","Open","High","Low","Close","Volume"] if c in df_aligned.columns]].copy()
df_feat = df_feat[[c for c in processed_cols if c in df_feat.columns]].copy()

In [56]:
# save PROCESSED ke data/processed
common_tag = f"{common_start.date()}_to_{common_end.date()}"

processed_common_path = os.path.join(DATA_PROCESSED_DIR, f"crypto_{interval}_{start}_COMMON_{common_tag}.csv")
processed_aligned_path = os.path.join(DATA_PROCESSED_DIR, f"crypto_{interval}_{start}_ALIGNED_{common_tag}.csv")
processed_feat_path = os.path.join(DATA_PROCESSED_DIR, f"crypto_{interval}_{start}_FEATURES_{common_tag}.csv")

df_common.to_csv(processed_common_path, index=False, date_format="%Y-%m-%d")
df_aligned.to_csv(processed_aligned_path, index=False, date_format="%Y-%m-%d")
df_feat.to_csv(processed_feat_path, index=False, date_format="%Y-%m-%d")

print("RAW saved      :", raw_path)
print("PROCESSED saved:", processed_common_path)
print("PROCESSED saved:", processed_aligned_path)
print("PROCESSED saved:", processed_feat_path)

print("\nProcessed summary:")
print(df_feat.groupby("Ticker")["Date"].agg(["min", "max", "count"]))
print("Missing Close (processed):", int(df_feat["Close"].isna().sum()))
print("Missing LogReturn (processed):", int(df_feat["LogReturn"].isna().sum()), "(normal: baris pertama tiap ticker NaN)")

RAW saved      : ../data/raw\crypto_yahoo_1d_2015-01-01_to_2025-12-19_LONG.csv
PROCESSED saved: ../data/processed\crypto_1d_2015-01-01_COMMON_2017-11-09_to_2025-12-19.csv
PROCESSED saved: ../data/processed\crypto_1d_2015-01-01_ALIGNED_2017-11-09_to_2025-12-19.csv
PROCESSED saved: ../data/processed\crypto_1d_2015-01-01_FEATURES_2017-11-09_to_2025-12-19.csv

Processed summary:
               min        max  count
Ticker                              
BNB-USD 2017-11-09 2025-12-19   2963
BTC-USD 2017-11-09 2025-12-19   2963
ETH-USD 2017-11-09 2025-12-19   2963
Missing Close (processed): 0
Missing LogReturn (processed): 3 (normal: baris pertama tiap ticker NaN)
